In [1]:
import pandas as pd

In [2]:
# Load Olist orders dataset
orders = pd.read_csv(
    "olist_orders_dataset.csv",
    parse_dates=[
        "order_purchase_timestamp",
        "order_approved_at",
        "order_delivered_carrier_date",
        "order_delivered_customer_date",
    ],
)

In [3]:
order_items = pd.read_csv("olist_order_items_dataset.csv")
products = pd.read_csv("olist_products_dataset.csv")
payments = pd.read_csv("olist_order_payments_dataset.csv")
translation = pd.read_csv("product_category_name_translation.csv")

In [5]:
# Explore different order statuses
print(orders["order_status"].value_counts())

order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64


In [6]:
# Cancellation rate

cancelled = orders[
    orders["order_status"] == "canceled"
]

cancel_rate = len(cancelled) / len(orders)

print("Cancellation count:", len(cancelled))
print("Cancellation rate:", cancel_rate)

Cancellation count: 625
Cancellation rate: 0.006285133898492574


In [7]:
# Join order items to products and English category names
items_with_english = order_items.merge(
    products[["product_id", "product_category_name"]],
    on="product_id",
    how="left"
).merge(
    translation,
    on="product_category_name",
    how="left"
)

In [8]:
# 1. Inter-arrival time distribution
orders_sorted = orders.sort_values("order_purchase_timestamp")
purchase_times = orders_sorted["order_purchase_timestamp"].dropna()

inter_arrival = purchase_times.diff().dt.total_seconds().dropna()
inter_arrival = inter_arrival[inter_arrival >= 0]

inter_arrival_samples = (
    inter_arrival.quantile([0.25, 0.50, 0.75])
    .round()
    .astype(int)
    .tolist()
)

print("\nInter-arrival time (seconds):")
print(inter_arrival.describe())


Inter-arrival time (seconds):
count    9.944000e+04
mean     6.714974e+02
std      1.917385e+04
min      0.000000e+00
25%      8.300000e+01
50%      2.220000e+02
75%      5.070000e+02
max      5.410280e+06
Name: order_purchase_timestamp, dtype: float64


In [9]:
# 2. Approval delay
approval_delay = (
    orders["order_approved_at"] - orders["order_purchase_timestamp"]
).dt.total_seconds().dropna()

approval_delay = approval_delay[approval_delay >= 0]

approval_samples = (
    approval_delay.quantile([0.25, 0.50, 0.75])
    .round()
    .astype(int)
    .tolist()
)

print("\nApproval delay (seconds):")
print(approval_delay.describe())


Approval delay (seconds):
count    9.928100e+04
mean     3.750874e+04
std      9.373681e+04
min      0.000000e+00
25%      7.740000e+02
50%      1.236000e+03
75%      5.249100e+04
max      1.623305e+07
dtype: float64


In [11]:
# 3. Dispatch delay
dispatch_delay = (
    orders["order_delivered_carrier_date"] - orders["order_approved_at"]
).dt.total_seconds().dropna()

dispatch_delay = dispatch_delay[dispatch_delay >= 0]

dispatch_samples = (
    dispatch_delay.quantile([0.25, 0.50, 0.75])
    .round()
    .astype(int)
    .tolist()
)

print("\nDispatch delay (seconds):")
print(dispatch_delay.describe())


Dispatch delay (seconds):
count    9.628500e+04
mean     2.470337e+05
std      3.022768e+05
min      1.500000e+01
25%      7.789300e+04
50%      1.599840e+05
75%      3.131180e+05
max      1.086589e+07
dtype: float64


In [12]:
# 4. Delivery delay
delivery_delay = (
    orders["order_delivered_customer_date"] - orders["order_delivered_carrier_date"]
).dt.total_seconds().dropna()

delivery_delay = delivery_delay[delivery_delay >= 0]

delivery_samples = (
    delivery_delay.quantile([0.25, 0.50, 0.75])
    .round()
    .astype(int)
    .tolist()
)

print("\nDelivery delay (seconds):")
print(delivery_delay.describe())


Delivery delay (seconds):
count    9.645200e+04
mean     8.064188e+05
std      7.567625e+05
min      0.000000e+00
25%      3.543718e+05
50%      6.134670e+05
75%      1.039430e+06
max      1.772850e+07
dtype: float64


In [13]:
# 5. Category distribution
category_counts = (
    items_with_english["product_category_name_english"]
    .dropna()
    .value_counts()
)

top_categories = category_counts.head(10).index.tolist()
print("Top categories:", top_categories)

Top categories: ['bed_bath_table', 'health_beauty', 'sports_leisure', 'furniture_decor', 'computers_accessories', 'housewares', 'watches_gifts', 'telephony', 'garden_tools', 'auto']


In [14]:
# 6. Order value distribution
order_values = payments.groupby("order_id")["payment_value"].sum()

order_value_samples = (
    order_values.quantile([0.25, 0.50, 0.75])
    .round(2)
    .tolist()
)

print("\nOrder value distribution:")
print(order_values.describe())


Order value distribution:
count    99440.000000
mean       160.990267
std        221.951257
min          0.000000
25%         62.010000
50%        105.290000
75%        176.970000
max      13664.080000
Name: payment_value, dtype: float64


In [15]:
# --- Final output: numbers to hardcode into Java workload generator ---
print("\n=== VALUES TO COPY INTO JAVA WORKLOAD GENERATOR ===")
print("inter_arrival_samples =", inter_arrival_samples)
print("approval_samples =", approval_samples)
print("dispatch_samples =", dispatch_samples)
print("delivery_samples =", delivery_samples)
print("order_value_samples =", order_value_samples)
print("top_categories =", category_counts.head(10).index.tolist())
print("cancel_rate =", cancel_rate)


=== VALUES TO COPY INTO JAVA WORKLOAD GENERATOR ===
inter_arrival_samples = [83, 222, 507]
approval_samples = [774, 1236, 52491]
dispatch_samples = [77893, 159984, 313118]
delivery_samples = [354372, 613467, 1039430]
order_value_samples = [62.01, 105.29, 176.97]
top_categories = ['bed_bath_table', 'health_beauty', 'sports_leisure', 'furniture_decor', 'computers_accessories', 'housewares', 'watches_gifts', 'telephony', 'garden_tools', 'auto']
cancel_rate = 0.006285133898492574
